# IGPO — One-Click Colab

Paper: [Wang et al., ICLR 2026](https://arxiv.org/abs/2510.14967) · Code: [ankknaiii/igpo-agentic-search](https://github.com/ankknaiii/igpo-agentic-search)

**Instructions**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Runtime → **Run all**

The next cell bootstraps the environment, runs integrity checks, executes a short IGPO training loop, and plots metrics.

In [ ]:
#@title ▶ Run all pipeline (bootstrap → check → train → plot)
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/ankknaiii/igpo-agentic-search.git"
ROOT = Path("/content/igpo-agentic-search")

def sh(cmd: str, check: bool = True):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=check)

# 1) Fresh sync to main (avoids stale Colab clones)
if ROOT.exists():
    sh(f"cd {ROOT} && git fetch origin main && git reset --hard origin/main", check=False)
else:
    sh(f"git clone --depth 1 {REPO_URL} {ROOT}")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# 2) Colab dependency hygiene
sh("pip uninstall -y torchao", check=False)
sh("pip install -q -U pip packaging")
sh("pip install -q -r requirements.txt")
sh("pip install -q -e .")
sh("pip uninstall -y torchao", check=False)

# 3) Integrity checks
sh("python scripts/integrity_check.py")
sh("pytest -q tests/")

# 4) Short training run (T4-friendly)
import torch
from igpo.train.trainer import TrainConfig, run_training

print("device=", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

cfg = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    model_source="auto",
    algo="igpo",
    max_steps=3,
    prompts_per_step=1,
    group_size=2,
    max_turns=2,
    max_new_tokens=64,
    ppo_epochs=2,
    gamma=0.95,
    learning_rate=1e-5,
    info_gain_type="prob_diff",
    info_gain_norm_mode="separate",
    output_dir="./outputs/igpo_colab",
    eval_every=0,
)
history = run_training(cfg)
print("final metrics:", history[-1])

# 5) Curves
import matplotlib.pyplot as plt
steps = [h.step for h in history]
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].plot(steps, [h.mean_f1 for h in history]); axes[0].set_title("mean F1")
axes[1].plot(steps, [h.collapse_rate for h in history]); axes[1].set_title("collapse rate")
axes[2].plot(steps, [h.mean_abs_ig for h in history]); axes[2].set_title("mean |IG|")
for ax in axes:
    ax.set_xlabel("step")
plt.tight_layout(); plt.show()

print("DONE. Artifacts in", (ROOT / cfg.output_dir).resolve())

## Optional: longer run / GRPO baseline

Only run after the pipeline cell succeeds.

In [ ]:
#@title Optional longer IGPO run
from igpo.train.trainer import TrainConfig, run_training

cfg_long = TrainConfig(
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    model_source="auto",
    algo="igpo",          # change to "grpo" for outcome-only baseline
    max_steps=20,
    prompts_per_step=2,
    group_size=4,
    max_turns=3,
    max_new_tokens=96,
    ppo_epochs=4,
    gamma=0.95,
    output_dir="./outputs/igpo_long",
    eval_every=10,
    eval_samples=20,
)
history_long = run_training(cfg_long)
history_long[-1]